# Ion inertial length in Jupiter's magnetosphere

In [ ]:
import numpy as np
import glob as glob
import datetime as dt
import matplotlib.pyplot as plt

from matplotlib.ticker import AutoMinorLocator, LogLocator, LinearLocator

from JunoPy.spice import InitSpiceKernels, get_juno_position
from JunoPy.mag import MagData, mva

## 1. Calculate the variation of density with radial distance. 

In [ ]:
r = np.linspace(6, 130, 100)

# Bagenal and Delamere (2011) model
a1 = 1987
a2 = 14
a3 = 0.05
b1 = -8.2
b2 = -3.2
b3 = -0.65

n = a1 * (r / 6)**(b1) + a2 * (r / 6)**(b2) + a3 * (r / 6)**(b3) 


mi = 16.6 * 1.67e-27
qi = 1.6e-19
rhoi = mi * n * 1e6
mu0 = 1.25e-6
e = 1.6e-19
di = 3e8 / np.sqrt(e**2 * n * 1e6 / mi / 8.84e-12) / 1e3 # / 71492e3
# di = mi / qi * np.sqrt(1 / rhoi / mu0) / 71492e3

## 2. Read the csv file and estimate spacecraft velocity

In [ ]:
def read_csv_file():
    start_times = []
    end_times = []
    durations = []
    with open('mva_epochs_selected.csv', 'r') as f:
        f.readline()
        for line in f:
            start_times.append(dt.datetime.strptime(line.split(',')[0], '%Y-%jT%H:%M:%S'))
            end_times.append(dt.datetime.strptime(line.split(',')[1], '%Y-%jT%H:%M:%S'))
            durations.append(float(line.split(',')[2]))
    return start_times, end_times, durations
            
start_times, end_times, durations = read_csv_file()
start_positions, _ = get_juno_position(start_times)
end_positions, _ = get_juno_position(end_times)

In [ ]:
N = len(start_times)

deltat = [(end_times[i] - start_times[i]).seconds for i in range(N)]
# dx = np.sqrt(np.sum(start_positions**2, axis=1) - np.sum(end_positions**2, axis=1))
dx = np.sqrt(np.sum((start_positions - end_positions)**2 , axis=1)) * 71492
v = dx / deltat * 1e3 # 71492e3
r_sc = np.sqrt(np.sum(start_positions**2, axis=1))
print(deltat)

In [ ]:
dx * 71492

## 3. Obtain the average magnetic field during the plasmoid interval.

In [ ]:
def get_average_B(start_times, end_times):
    """
    """
    B = []
    
    
    for start, end in zip(start_times, end_times):    
        magfile = glob.glob('data/mag/*{:s}ss*.hdf5'.format(start.strftime('%Y%j')))[0]
        magdata = MagData(filename=magfile)
        magdata.df = magdata.df.loc[start:end]
        
        B.append(magdata.df['Bmag'].mean())
        
        
    return np.asarray(B)

B_avg = get_average_B(start_times, end_times)

In [ ]:
def get_deltaB(start_times, end_times):
    """
    """
    dB = []
    
    
    for start, end in zip(start_times, end_times):    
        magfile = glob.glob('data/mag/*{:s}ss*.hdf5'.format(start.strftime('%Y%j')))[0]
        magdata = MagData(filename=magfile)
        magdata.df = magdata.df.loc[start:end]
        
        dB.append(magdata.df['Bmag'].max()**2 - magdata.df['Bmag'].min()**2)
        
        
    return np.asarray(dB) * 1e-18 / 2 / (4 * np.pi * 1e-7) * 1e9

deltaB = get_deltaB(start_times, end_times)

## Plot the data.

In [ ]:
# fig, ax = plt.subplots(3, 1, sharex=True, dpi=600, figsize=(5,5), )

fig = plt.figure(figsize=(5,5), dpi=600)
gs = fig.add_gridspec(4, 4, hspace=0.1, wspace=0.12, left=0.25)

ax = []
ax.append(fig.add_subplot(gs[0, :-1]))
ax.append(fig.add_subplot(gs[1, :-1]))
ax.append(fig.add_subplot(gs[2, :-1]))
ax.append(fig.add_subplot(gs[3, :-1]))

ax.append(fig.add_subplot(gs[1, -1]))   # Histogram axes
ax.append(fig.add_subplot(gs[2, -1]))
ax.append(fig.add_subplot(gs[3, -1]))

# 1. Density versus Radial distance
ax[0].semilogy(r, n, color='k', zorder=1, lw=1.)
ax[0].semilogy(r, n / 10, color='k', ls='dashed', zorder=1, lw=1.)
ax[0].semilogy(r, n * 10, color='k', ls='dotted', zorder=1, lw=1.)
ax[0].fill_between(r, n/10, y2=n*10, color=[0.5, 0.5, 0.5], alpha=0.2, zorder=1)
ax[0].set_ylabel('n\n[cm$^{-3}$]')
ax[0].yaxis.set_major_locator(LogLocator(base=10, numticks=8))
ax[0].legend(['$n$ - Density profile\nBagenal and Delamere (2011)',
             '$n/10$',
             '$10n$'], prop={'size': 5}, framealpha=1.0)

# 2. Ion inertial length
ax[1].semilogy(r, di, color='k', zorder=1, lw=1.)
ax[1].semilogy(r, di * np.sqrt(10), color='k', ls='dashed', zorder=1, lw=1.)
ax[1].semilogy(r, di / np.sqrt(10), color='k', ls='dotted', zorder=1, lw=1.)
ax[1].scatter(r_sc, dx, marker='x', zorder=2, color='k', s=8.0)
ax[1].scatter(r_sc, dx + 100 * np.asarray(deltat), marker='x', zorder=2, s=8.0, color=[0.2, 0.2, 0.8])
ax[1].scatter(r_sc, dx + 500 * np.asarray(deltat), marker='x', zorder=2, s=8.0, color=[0.8, 0.2, 0.2])
ax[1].fill_between(r, di / np.sqrt(10), y2=di * np.sqrt(10), color=[0.5, 0.5, 0.5], alpha=0.2, zorder=1)
ax[1].set_ylabel('$\Delta x + |v\Delta t|$\n[km]')
ax[1].yaxis.set_major_locator(LogLocator(base=10, numticks=8))

ax[1].legend(['$d_i$',
              '$\sqrt{10}d_i$',
              '$d_i/\sqrt{10}$',
              'v=0',
              'v=100 km/s',
              'v=500 km/s'], loc='lower left', prop={'size': 6}, framealpha=1.0,
              bbox_to_anchor=(1.02, 1.0, 0.5, 0.5))

# 3. Alfven speed
n_sc = a1 * (r_sc / 6)**(b1) + a2 * (r_sc / 6)**(b2) + a3 * (r_sc / 6)**(b3) 
vAlfven = B_avg / np.sqrt(mu0 * mi * n_sc * 1e6) * 1e-9 / 1e3

ax[2].scatter(r_sc, vAlfven, marker='x', zorder=2, color='k', s=8.0)
ax[2].set_ylabel('v$_A$\n[km/s]')
ax[2].set_yticks([0, 200, 400, 600, 800])
ax[2].text(125, 450, 
           'N = {:d}\nMean {:.2f} km/s\nMedian {:.2f} km/s'.format(
                N, np.mean(vAlfven), np.median(vAlfven)), 
           size=7, horizontalalignment='right')


# 4. Delta B 
ax[3].scatter(r_sc, deltaB, marker='x', zorder=2, color='k', s=8.0)
ax[3].set_ylabel('$\Delta p_B$ [nPa]')
ax[3].set_xlabel('r [R$_J$]')
# ax[3].set_yticks([0, 2, 4, 6])
ax[3].text(125, 0.035, 
           'N = {:d}\nMean {:.3f} nPa\nMedian {:.3f} nPa'.format(
                N, np.mean(deltaB), np.median(deltaB)), 
           size=7, horizontalalignment='right')
ax[3].set_xlabel('r [R$_J$]')

# Histograms.
logbins = np.logspace(np.log10(ax[1].get_ylim()[0]),np.log10(ax[1].get_ylim()[1]), 20)
ax[4].hist([dx, 
            dx + 100 * np.asarray(deltat),
            dx + 500 * np.asarray(deltat)], 
           orientation='horizontal', bins=logbins, 
           histtype='step', color=['k', [0.2,0.2,0.8], [0.8,0.2,0.2]])
ax[4].set_yscale('log')

ax[5].hist(vAlfven, orientation='horizontal', histtype='step', color='k')
ax[6].hist(deltaB, orientation='horizontal', histtype='step', color='k')

# Cleanup
for a in ax[0:4]:
    a.set_xlim([r[0], r[-1]])
    a.xaxis.set_minor_locator(AutoMinorLocator())
    a.grid('on', which='minor', lw=0.5, ls='dotted')
    a.grid(color='k', ls='dashed', lw=0.25)
    a.label_outer()
    
for a in ax[4:]:
    a.set_xlabel('# Events')
    a.set_yticks([])
    a.set_xticks([4, 8])
    a.set_xlim([0, 10])
    a.label_outer()

ax[0].annotate("(a)", xy=(0.05, 0.80), xycoords="axes fraction")
ax[1].annotate("(b)", xy=(0.05, 0.80), xycoords="axes fraction")
ax[2].annotate("(c)", xy=(0.05, 0.80), xycoords="axes fraction")
ax[3].annotate("(d)", xy=(0.05, 0.80), xycoords="axes fraction")


fig.tight_layout()
fig.savefig('out_images/ion_inertial_selected.png')

# plt.close(fig)

## Get average B_1

In [ ]:
def get_average_mva(start_times, end_times):
    """
    """
    B1_avg = []
    ratio_eigval_32 = []
    ratio_eigval_21 = []
    
    for start, end in zip(start_times, end_times):    
        magfile = glob.glob('data/mag/*{:s}ss*.hdf5'.format(start.strftime('%Y%j')))[0]
        magdata = MagData(filename=magfile)
        magdata.df = magdata.df.loc[start:end]
        magdata = mva(magdata)
        
        B1_avg.append(magdata.df['B1'].mean())
        ratio_eigval_32.append(magdata.eigval[2] / magdata.eigval[1])
        ratio_eigval_21.append(magdata.eigval[1] / magdata.eigval[0])
        
    return np.asarray(B1_avg), np.asarray(ratio_eigval_32), np.asarray(ratio_eigval_21)

B1_avg, ratio_eigval_32, ratio_eigval_21 = get_average_mva(start_times, end_times)

In [ ]:
def plot_histogram(plotvars):
        
    # Create new figure and plot
    fig, ax = plt.subplots(3, 1, dpi=600, figsize=(6,8))
    

    for a, var in zip(ax, plotvars):
        a.hist(var, bins=100, color=[0.5, 0.5, 0.5], ec='k', zorder=2)
        a.xaxis.set_minor_locator(AutoMinorLocator())
        a.yaxis.set_minor_locator(AutoMinorLocator())
        a.set_ylabel('n', weight='bold')
    
#     ax[0].set_xlabel('<|B$_1$|> [nT]')
#     ax[1].set_xlabel('$\lambda_3 / \lambda_2$')
#     ax[2].set_xlabel('$\lambda_2 / \lambda_1$')
    
    ax[0].annotate("(a) <|B$_1$|> [nT] ", xy=(0.7, 0.8), xycoords="axes fraction")
    ax[1].annotate("(b) $\lambda_3 / \lambda_2$", xy=(0.7, 0.8), xycoords="axes fraction")
    ax[2].annotate("(c) $\lambda_2 / \lambda_1$", xy=(0.7, 0.8), xycoords="axes fraction")
    
    ax[0].set_xlim([0, 9])
    ax[1].set_xlim([0, 100])
    ax[2].set_xlim([0, 300])
    
    plt.show()
    fig.savefig('out_images/hist_mva.png')
    plt.close(fig)
    
plot_histogram([np.abs(B1_avg), ratio_eigval_32, ratio_eigval_21])